# LAB 8: 스테이지와 반정형 데이터

## Snowflake 스테이지 오브젝트 생성

👉 이 실습에서는 Education Services 팀이 실습용으로 미리 생성해 둔 **S3**(Amazon Simple Storage Service) 버킷을 가리키는 스테이지를 생성합니다. 

시작하기에 앞서, 이 실습 전반에서 사용할 **컨텍스트 정보**를 가져오겠습니다. 

- **Start** 버튼을 클릭하여 이 노트북을 활성화하세요.

- 다음 Python 셀을 실행하세요.


#### :warning: 이 노트북에 대해 새 세션이 시작될 때마다, 후속 셀에서 사용할 '변수'를 구성하기 위해 아래 셀을 다시 실행해야 합니다. :warning:


In [ ]:
import streamlit as st
from snowflake.snowpark.context import get_active_session
session = get_active_session()
user = session.get_current_user().strip('"')
your_db = user + '_DB'
print('현재 CONTEXT 정보:')
print('---------------------------------')
print(session)
print('현재 USER는 ' + user)

### 스테이지 생성 🥋

1. Snowsight 오브젝트 브라우저에서 **Catalog** > **Database Explorer**로 이동하여 이전 실습에서 생성한 **(animal)_UTIL_DB** 데이터베이스를 선택하세요.
1. 그런 다음 **PUBLIC**이라는 스키마를 선택하세요.
1. 오른쪽 상단의 파란색 **Create** 버튼을 클릭하세요.
1. **Stage** > **External Stage** > **Amazon S3**를 선택하세요.

![스테이지 생성 옵션(이미지)](https://edu-cdev-images.s3.us-west-2.amazonaws.com/ob/ob_create_stage_new_1.png)

스테이지 생성 대화 상자가 나타납니다.

![스테이지 생성 대화 상자(이미지)](https://edu-cdev-images.s3.us-west-2.amazonaws.com/ob/ob_create_stage_dialog_1.png)

1. **Stage Name** 텍스트 상자에 `like_a_window_into_an_s3_bucket`을 입력하세요.
1. **URL** 텍스트 상자에 `s3://uni-lab-files`를 입력하세요.
1. **Directory table** 옵션이 선택된 상태를 유지하세요.
1. 오른쪽 하단 모서리에 있는 파란색 **Create** 버튼을 클릭하세요.
1. 다음 화면에서 사용할 Snowflake 가상 웨어하우스의 이름을 입력하라는 메시지가 표시될 수 있습니다. 자신의 동물 이름이 지정된 웨어하우스 **(animal)_WH**를 선택하세요.


### 오브젝트 브라우저에는 새로 생성한 새 스테이지에서 사용 가능한 파일들이 표시됩니다. 📓

![스테이지 파일(이미지)](https://edu-cdev-images.s3.us-west-2.amazonaws.com/ob/ob_new_stage_files.png)


💡 **팁**: 이 파일과 폴더들은 이 과정을 제공하는 Snowflake Education Services 팀이 소유하고 관리하는 AWS S3 버킷에 저장되어 있습니다. 여기에서 파일 목록을 볼 수 있는 이유는, 여러분이 스테이지를 생성했기 때문인데, 이 스테이지는 해당 버킷 안의 파일을 보고 액세스할 수 있도록 해주는 창과 같은 역할을 합니다. 우리의 버킷은 공개되어 있지만, 여러분 회사에서 생성하는 버킷은 대부분 자격 증명이 필요할 것입니다.


### 스테이지인가, 아닌가? 📓 

Snowflake에서 스테이지를 사용할 때 헷갈릴 수 있는 부분 중 하나는, 여러분이 생성한 스테이지 오브젝트가 실제 위치가 아니라는 점입니다. 파일이 저장되어 있는 위치(즉, 파일을 보관하고 있는 S3 버킷)는 이미 존재하고 있었습니다. 그렇다면 여러분은 방금 무엇을 만든 걸까요? 

여러분은 Snowflake에게 '이미 파일이 스테이징되어 있는 어떤 위치에 대한 정보'를 알려주는 무언가를 만든 것입니다. 실제 스테이지 위치를 만든 것이 아니라, 그 스테이지 위치를 들여다볼 수 있는 창 같은 것을 만든 것입니다. Snowflake 스테이지 오브젝트는 File format이 파일을 로드하는 작업을 쉽게 만들어주는 구성 정보를 보유하고 있다는 점에서 File Format과 비슷한 역할을 합니다.

때로는 Snowflake 스테이지를 정의할 때 액세스 자격 증명을 함께 제공하기도 하지만, 이번 경우에는 그렇지 않았습니다. 이번에 만든 스테이지는 이미 파일이 스테이징되어 있는 S3 버킷을 가리키도록 이름을 붙인 하나의 오브젝트일 뿐입니다.  


## SQL 셀에서 `LIST` 명령 사용하기 🥋 

### `LIST` 명령을 사용하여 새로 생성한 스테이지에 있는 파일을 확인하세요. 🥋 

`LIST` 명령은 Snowflake 스테이지에 스테이징된 파일들의 목록을 반환합니다(즉, 로컬 파일 시스템에서 업로드되었거나 테이블에서 언로드된 파일들). 
- 이 명령은 `LS`로 축약해서 사용할 수도 있습니다.
- Snowflake에서 스테이지 오브젝트를 참조할 때는, 이름 앞에 앰퍼샌드(`@`) 기호를 붙여 사용합니다.

아래의 `LIST` 명령어를 직접 실행해 보세요.


In [ ]:
USE SCHEMA {{user}}_UTIL_DB.PUBLIC;

LIST @like_a_window_into_an_s3_bucket;

### Snowflake 오브젝트 명명 규칙 📓

Snowflake는 대소문자를 구분하지 않습니다. 우리는 새로 생성한 스테이지 오브젝트의 이름을 소문자로 입력했습니다. Snowflake는 사용자가 모든 것을 **대문자**로 입력하려는 의도라고 가정하므로 자동으로 변환해 줍니다. 따라서 오브젝트를 생성하거나 조회할 때 소문자나 대소문자가 섞인 형태로 입력해도, Snowflake는 내부적으로 모두 대문자로 처리합니다.

(단, 오브젝트를 생성할 때 따옴표를 사용하면 이야기가 달라집니다. 이 경우 이후에도 해당 오브젝트를 사용할 때 반드시 따옴표를 계속 사용해야 합니다.)

따라서 스테이지에 대해 명령을 실행할 때는 어떤 대소문자 표기를 사용해도 정상적으로 동작합니다. 하지만 S3는 매우 특별합니다. 스테이지 오브젝트 이름을 지난 이후부터는 반드시 정확한 철자를 사용해야 합니다. 파일 확장자까지 포함하여 대소문자가 정확히 일치해야 합니다.  

![스테이지 파일(이미지)](https://edu-cdev-images.s3.us-west-2.amazonaws.com/ob/ob_list_command_2.png)


### :mag_right: Check 8 (OB08) 🔎

- `like_a_window_into_an_s3_bucket`라는 이름의 External Stage가 생성되어 있나요?
- 작업 결과 확인을 위해 채점용 Stored Procedure를 호출하세요.


In [ ]:
CALL common_db.resources.local_grader('OB08', '{{user}}');

## `COPY INTO` 구문을 사용하여 데이터를 로드하기 🥋 

### 토양 유형(Soil Type)을 위한 테이블을 생성하세요. 🥋 

반드시 **(animal)_GARDEN_PLANTS** 데이터베이스의 **VEGGIES** 스키마에 생성해야 합니다. 이를 위해 다음 SQL 셀의 2번째 줄을 수정해야 합니다. 해시`('#')` 기호를 올바른 스키마 이름으로 대체한 후 셀을 실행하세요.


In [ ]:
-- 다음 줄의 해시 문자('#')를 변경하세요
USE SCHEMA {{user}}_GARDEN_PLANTS.#######;

CREATE OR REPLACE TABLE vegetable_details_soil_type
( plant_name VARCHAR(25)
 ,soil_type NUMBER(1,0)
);

### S3 버킷의 파일을 새 테이블로 로드하세요. 📓

이전에 Snowsight의 **데이터 로드** 화면을 사용하여 스테이징된 파일의 데이터를 테이블로 복사했습니다. 이러한 '마법사 기반' 방식에서는 Snowflake가 이 작업을 수행하기 위한 코드를 내부적으로 자동 생성하고 실행합니다. 이번 데이터 로드에서는 프로그래밍 방식을 살펴볼 것입니다.

노트북의 SQL 셀에서 실행하는 `COPY INTO` 구문을 사용하게 됩니다.

`COPY INTO` 구문을 사용하려면 다음 네 가지 요소가 준비되어 있어야 합니다.

1. 테이블 

1. Stage 오브젝트

1. 파일

1. File Format(선택 사항)

**File Format**은 선택 사항입니다. 대안이 존재하기 때문이지만, File Format을 지정하면 프로세스가 더 깔끔해집니다. 앞서 언급한 바와 같이, File Format은 스테이지에서 로드되는 데이터를 처리하는 방법에 대한 지침을 Snowflake에 제공하는 오브젝트입니다. 다음 예시에서는 File Format을 별도로 정의하지 않고 이러한 지침을 인라인 방식으로 제공할 것입니다.


### 실행 가능한 `COPY INTO` 구문 🥋 


In [ ]:
COPY INTO vegetable_details_soil_type
FROM @{{user}}_util_db.public.like_a_window_into_an_s3_bucket
FILES = ( 'VEG_NAME_TO_SOIL_TYPE_PIPE.txt')
FILE_FORMAT = (
    TYPE=csv
    FIELD_DELIMITER = '|'
    SKIP_HEADER=1
);

### :mag_right: Check 9 (OB09) 🔎

- **(animal)_garden_plants.veggies** 스키마의 **vegetable_details_soil_type** 테이블에 42개의 로우가 로드되어 있나요?
- 작업 결과 확인을 위해 채점용 Stored Procedure를 호출하세요.


In [ ]:
CALL common_db.resources.local_grader('OB09', '{{user}}');

## 데이터 로딩 팁과 요령 📓

- 모든 평면 파일(flat file)은 CSV(쉼표로 구분된 값) 형식의 File Format을 사용하여 로드됩니다. 따라서 TSV, 파이프 구분, .txt 등 모든 평면 파일(flat file)에 대해 `TYPE = CSV`를 사용하세요.

- **FIELD_DELIMITER** 속성은 매우 중요합니다. 파일에서 실제로 사용되는 컬럼 구분자와 일치해야 합니다. 

- Data Load Wizard를 사용하면 File Format(명명된 형식 또는 인라인 형식) 작성에 도움이 됩니다. 드롭다운 목록에서 필요한 설정들을 선택하기만 하면 됩니다. 그 다음 Show SQL 링크를 클릭하면 해당 설정이 반영된 SQL 코드를 확인할 수 있습니다. 


## 챌린지 실습: 토양 유형(Soil Type) 조회 테이블 생성 🎯 

이 챌린지 실습에서는 **lu_soil_type**이라는 새 테이블을 생성하고, 제공된 파일의 데이터를 다음 방법 중 하나로 로드해야 합니다.
- **Load Data Wizard** 사용 **또는**
- 직접 `COPY INTO` 구문 작성 및 실행

:warning: 이번 연습에서는 개념 수준의 지침만 제공되며, 지금까지 학습한 내용을 바탕으로 단계들을 '스스로 파악해' 진행해야 합니다. :warning:

먼저 테이블 생성 작업을 시작하세요. 반드시 **(animal)_GARDEN_PLANTS** 데이터베이스의 **VEGGIES** 스키마에 생성해야 합니다. 


In [ ]:
USE SCHEMA {{user}}_GARDEN_PLANTS.VEGGIES;

CREATE OR REPLACE TABLE lu_soil_type(
    soil_type_id NUMBER,	
    soil_type VARCHAR(15),
    soil_description VARCHAR(75)
);

### 소스 데이터 파일을 다운로드하세요. 🎯 

다음 Python 코드 셀을 실행하고 생성된 링크를 클릭하여 **LU_SOIL_TYPE.tsv** 파일을 다운로드하세요.


In [ ]:
snowpark_df = session.sql("SELECT GET_PRESIGNED_URL(@common_db.resources.course_files, 'LU_SOIL_TYPE.tsv')")
collected_data = snowpark_df.collect()
st.write('다음 링크를 클릭하여 파일을 다운로드하세요:')
st.write(collected_data[0][0])

### 다운로드한 파일에서 테이블 로우를 로드하세요. 🎯

**LU_SOIL_TYPE.tsv** 파일은 이전에 로드한 **VEG_NAME_TO_SOIL_TYPE_PIPE.txt** 파일과 많은 File Format 속성을 공유하지만, 한 가지 주요 예외가 있습니다. 이를 요약해 보겠습니다.
- **TYPE**: 파일 확장자가 **.tsv**입니다. 따라서 파일 [유형](https://docs.snowflake.com/ko/sql-reference/sql/copy-into-table#type-csv)을 파악할 수 있을 것입니다.
- **SKIP_HEADER**: 파일에는 헤더 로우가 하나 있습니다.
- **FIELD_DELIMITER**: 파일은 파이프(`'|'`)로 구분된 것이 **아니라** **TAB**으로 구분되어 있습니다. TAB 구분자는 `'\t'`라는 문자로 표현됩니다.


#### 옵션 1:
- Snowsight의 **Load Data Wizard**를 사용하세요.

#### 또는

#### 옵션 2:
- 위의 설명을 참고하여 다음 SQL `COPY INTO` 구문 조각을 수정하고 실행하세요.
- 대상 테이블의 이름을 입력하세요(라인 1).
- File Format 옵션을 입력하세요(라인 5-7).


In [ ]:
COPY INTO ##_####_####
FROM @{{user}}_util_db.public.like_a_window_into_an_s3_bucket
FILES = ( 'LU_SOIL_TYPE.tsv')
FILE_FORMAT = (
    ####=###
    #####_######### = ###
    ####_######=#
    FIELD_OPTIONALLY_ENCLOSED_BY = '"'
);

### :mag_right: Check 10 (OB10) 🔎

- **(animal)_garden_plants.veggies** 스키마의 **lu_soil_type** 테이블에 8개의 로우가 로드되어 있나요?
- 작업 결과 확인을 위해 채점용 Stored Procedure를 호출하세요.


In [ ]:
CALL common_db.resources.local_grader('OB10', '{{user}}');

## 반정형 데이터로 작업하기 📓

반정형 데이터는 전통적인 정형 데이터의 기준에는 완전히 부합하지 않지만, 데이터 안에 개별적이고 구분 가능한 엔터티를 식별할 수 있도록 하는 태그(레이블) 또는 기타 마크업 정보를 포함하고 있는 데이터입니다. 반정형 데이터를 정형 데이터와 구분 짓는 두 가지 핵심 특징은 '중첩된 데이터 구조'와 '고정된 스키마의 부재'입니다.

- 반정형 데이터는 사전에 스키마를 정의할 필요가 없으며, 지속적으로 진화할 수 있습니다(새로운 속성이 언제든 추가될 수 있음).
- 정형 데이터가 평면 테이블 형태로 데이터를 표현하는 것과 달리, 반정형 데이터는 N-계층의 중첩된 정보 구조를 가질 수 있습니다.

아래는 Snowflake에서 지원하는 반정형 데이터 유형 중 하나인 JSON 데이터의 예시입니다.

![식물 세부 정보 테이블 데이터(이미지)](https://edu-cdev-images.s3.us-west-2.amazonaws.com/ob/ob_json_data_extract.png)

### `VARIANT` 데이터 유형

Snowflake는 [`VARIANT`](https://docs.snowflake.com/ko/sql-reference/data-types-semistructured#label-data-type-variant) 데이터 유형을 제공하여 반정형 데이터의 저장을 지원합니다. 이 데이터 유형은 반정형 데이터와 함께 자주 사용되는 [`ARRAY`](https://docs.snowflake.com/ko/sql-reference/data-types-semistructured#array)와 [`OBJECT`](https://docs.snowflake.com/ko/sql-reference/data-types-semistructured#object)를 포함하여, 다른 어떤 데이터 유형의 값도 저장할 수 있습니다. 이 데이터 유형을 사용하면 Snowflake로 반정형 데이터를 수집할 때 계층적 또는 중첩된 형식을 유지할 수 있습니다.


### `VARIANT` 컬럼을 포함하는 테이블을 생성하세요. 🥋

**VEGETABLE_DETAILS_PLANT_HEIGHT**라는 이름의 테이블을 **(animal)_GARDEN_PLANTS** 데이터베이스의 **VEGGIES** 스키마에 생성하세요. 이 테이블은 `VARIANT` 데이터 유형의 단일 컬럼만 포함합니다. 


In [ ]:
CREATE OR REPLACE TABLE {{user}}_GARDEN_PLANTS.VEGGIES.VEGETABLE_DETAILS_PLANT_HEIGHT (
	record VARIANT
);

### JSON 소스 데이터 파일 다운로드하세요. 🥋

다음 Python 코드 셀을 실행하고 생성된 링크를 클릭하여 JSON **veg_plant_height.json** 파일을 다운로드하세요.

사용 중인 브라우저에 따라 새 탭 또는 새 윈도우에서 바로 열리거나 다운로드될 수 있습니다. 다운로드된 경우, 로컬 시스템의 텍스트 편집기에서 파일을 열어 구조를 검토하세요.

💡 **팁**: 이 파일은 구조와 내용을 검토하기 위해 다운로드하는 것입니다. 이 파일에서 데이터를 로드할 때는 스테이지에 이미 위치한 파일을 사용할 것입니다.


In [ ]:
snowpark_df = session.sql("SELECT GET_PRESIGNED_URL(@common_db.resources.course_files, 'veg_plant_height.json')")
collected_data = snowpark_df.collect()
st.write('다음 링크를 클릭하여 파일을 다운로드하세요:')
st.write(collected_data[0][0])

### JSON 데이터를 로드하기 위해 실행할 수 있는 `COPY INTO` 구문입니다. 🥋 

이제 다음 구문을 실행하여 이 JSON 데이터를 새 테이블에 로드하세요.

- 다음 **File Format** 사양에서, 정형 데이터가 아니라 반정형 데이터를 로드하기 위해 무엇이 변경되었나요?


In [ ]:
COPY INTO vegetable_details_plant_height
FROM @common_db.resources.course_files/veg_plant_height.json
FILE_FORMAT = (TYPE = 'JSON');

### JSON 데이터에서 로드된 테이블을 쿼리하세요. 🥋

Snowflake에는 `VARIANT` 타입으로 저장된 복잡한 계층적 데이터를 쿼리하기 위한 특수 연산자와 함수들이 있습니다. 이에 대해서는 곧 살펴보겠지만, 우선은 정형 데이터 세트에 사용하는 것과 같은 유형의 일반 SQL 구문을 사용하여 **vegetable_details_plant_height** 테이블의 데이터 한 로우를 검토해 보겠습니다.


In [ ]:
SELECT *
FROM vegetable_details_plant_height
LIMIT 1;

### 반정형 데이터를 쿼리합니다. 

위 쿼리에서 **RECORD**라는 `VARIANT` 컬럼에 네 개의 키 페어 값이 포함되어 있는 것을 관찰할 수 있습니다. 이는 데이터가 수집된 형식 그대로 유지되는 것이므로 문제가 없습니다. 하지만 개별 엘레멘트를 분리하여 처리하고 보고를 위해 정형화된 형식으로 사용해야 하는 경우가 있을 수 있습니다. 

다음은 Snowflake에서 반정형 데이터를 **탐색(traversing)**하기 위한 몇 가지 주요 지침입니다. 
- `VARIANT` 컬럼 이름과 첫 번째 수준 엘레멘트 사이에 콜론(`:`)을 삽입합니다.
    - <column>:<level1_element>
- JSON 오브젝트에서 계층 구조의 더 하위에 중첩된 엘레멘트에 액세스하려면 점 표기법(dot notation)을 사용하세요. 
    - <column>:<level1_element>.<level2_element>.<level3_element>
- **컬럼 이름**은 대소문자를 구분하지 않아도 되나 엘레멘트 이름은 **구분합니다**.
- 엘레멘트 이름은 선택적으로 큰따옴표로 묶을 수 있습니다.

**vegetable_details_plant_height** 테이블의 로우를 **평면화(flatten)**(즉, 정형 데이터로 표현)하는 간단한 이 예시를 시도해 보세요.


In [ ]:
//정규화된 테이블처럼 보이도록 데이터를 반환합니다.
SELECT 
    record:PLANT_NAME::STRING AS plant_name,
    record:UOM AS uom, -- 캐스팅 없음
    record:LOW_END_OF_RANGE::INTEGER AS low_end_of_range,
    record:HIGH_END_OF_RANGE::INTEGER AS high_end_of_range
FROM vegetable_details_plant_height;

### `VARIANT` 값 캐스팅 📓

위의 쿼리를 실행하면서 눈치챘을 수도 있겠지만, **UOM** 컬럼의 값은 다른 컬럼들과 달리 **캐스팅**(다른 데이터 유형으로 변환)이 이루어지지 않아 조금 다르게 보였을 것입니다. 이 값은 큰따옴표로 감싸져 있었습니다(예: "F").

이는 해당 컬럼 값이 `VARCHAR` 또는 그 동의어인 `STRING` 타입이라는 뜻이 아니라, 여전히 `VARIANT` 값이라는 것을 의미합니다. `VARIANT` 값 자체가 문자열인 것은 아니며, `VARIANT` 값은 문자열을 포함하고 있습니다.

Snowflake SQL에서는 다음과 같은 방법으로 데이터 유형을 캐스팅할 수 있습니다.
- [`CAST()`](https://docs.snowflake.com/ko/sql-reference/functions/cast) 함수 사용
- 대체 구문으로 `::` 연산자 사용(예: **record:PLANT_NAME::STRING**)


## 챌린지 실습: 반정형 데이터를 보여주는 뷰 생성하기 🎯 

이 챌린지 실습에서는 새 [뷰](https://docs.snowflake.com/ko/user-guide/views-introduction) 오브젝트를 생성합니다. 뷰를 사용하면 쿼리 결과를 마치 테이블처럼 액세스할 수 있습니다. 목표는 **vegetable_details_plant_height**의 데이터를 정규화된 방식으로 표시하는 것입니다.

1. 아래 셀에 있는 템플릿 SQL을 수정하세요. 수정이 필요한 세 줄은 `-- ***`로 표시되어 있습니다.
1. **UOM** 컬럼을 `VARCHAR` 타입으로 **캐스팅**하세요.
1. **LOW_END_OF_RANGE** 컬럼과 **HIGH_END_OF_RANGE** 컬럼의 순서를 바꾸세요(**SWAP**).


In [ ]:
-- 위의 지시사항에 따라 아래 코드를 수정한 후 실행하여 오브젝트를 생성하세요
CREATE OR REPLACE VIEW vegetable_details_plant_height_vw AS 
SELECT 
    record:PLANT_NAME::STRING AS plant_name,
    record:UOM AS uom, -- ***
    record:LOW_END_OF_RANGE::INTEGER AS low_end_of_range, -- ***
    record:HIGH_END_OF_RANGE::INTEGER AS high_end_of_range -- ***
FROM vegetable_details_plant_height;

In [ ]:
-- 그런 다음 새 뷰에 대해 이 쿼리를 실행하여 출력을 확인합니다
SELECT *
FROM vegetable_details_plant_height_vw;

### :mag_right: Check 11 (OB11) 🔎

- **vegetable_details_plant_height_vw**라는 이름의 뷰를 **(animal)_GARDEN_PLANTS** 데이터베이스의 **VEGGIES** 스키마에 생성했나요?
- **UOM**이 `VARCHAR`(텍스트) 컬럼으로 캐스팅되었으며, **HIGH_END_OF_RANGE** 컬럼이 세 번째, **LOW_END_OF_RANGE** 컬럼이 네 번째 위치에 있나요?
- 작업 결과 확인을 위해 채점용 Stored Procedure를 호출하세요.


In [ ]:
CALL common_db.resources.local_grader('OB11', '{{user}}');

## 지식 테스트 :mag_right:

아래의 대화형 퀴즈 문제를 통해 이해도를 확인해 보세요. 각 `RUN_THIS_QUIZ_QUESTION_` 셀에는 Snowflake 기능과 관련된 객관식 문제를 제시하는 Streamlit 위젯이 포함되어 있습니다.  

**지침:**  
1. 노트북 셀 위에 커서를 올려 추가 컨트롤을 표시하세요.
1. 각 퀴즈 셀 오른쪽의 ▶️ **Play 버튼**을 클릭하여 실행하세요.  
1. 제공된 옵션에서 답을 선택하세요.  
1. 다음으로 넘어가기 전에 피드백을 검토하세요. 

💡 **참고:** 궁금하시면 셀을 확장하여 코드를 볼 수 있지만, 필수는 아닙니다. 이 퀴즈들은 필수 사항이 아닙니다. 배운 내용을 복습하며 연습할 기회를 제공하기 위한 것입니다.  


In [ ]:
st.divider()
question = "Snowflake Stage 오브젝트에 LIKE_A_WINDOW_INTO_AN_S3_BUCKET이라는 이름을 지어준 이유가 무엇이라고 생각하나요?"
options = ["아래 선택을 고르세요...",
           "A) 짧기 때문에 입력하기 쉽습니다", 
           "B) 스테이지가 쉽게 손상될 수 있기 때문입니다. 창문이 때때로 그렇듯이", 
           "C) \"stage\"라는 단어가 위치를 의미하기 때문에, 이 경우 위치는 S3 버킷이지, 우리가 생성한 Snowflake 오브젝트가 아닙니다"]                                 

user_answer = st.radio(question, options, index=0)
if user_answer:
    if user_answer == "아래 선택을 고르세요...": # 이 옵션은 streamlit이 1.26.0 이상으로 업그레이드될 때까지의 임시 방편입니다. 그래서 index=None을 사용할 수 있습니다.
        ''
    else:
        answer = '8ffd32bf2db7cdd1dabd7d19ab5d12ca'
        # 옵션 문자 (A, B, C, 또는 D)를 추출하세요
        selected_option = user_answer.split(')')[0] + ')'
        response = session.sql(f"call common_db.resources.quiz_temp('{answer}', '{user_answer}', 'False')").collect()
        if response:
            value = response[0]['QUIZ_TEMP']
        st.write(f"{selected_option} {value}")

In [ ]:
st.divider()
question = "대문자, 소문자, 혼합 대소문자? 누가 신경 쓸까요? 아래 문장 중 Snowflake의 대소문자 구분에 대해 참인 것은 무엇인가요?"
options = ["아래 선택을 고르세요...",
           "A) 이중 따옴표로 묶지 않는 한 Snowflake는 항상 대문자로 입력하려고 했다고 가정합니다", 
           "B) 만약 HAPPY라는 이름의 테이블을 생성하면 Snowflake는 실제로 그 이름을 happy로 지정합니다", 
           "C) \"hAppY\"라는 이름의 테이블을 생성하면, 쿼리할 때마다 단일 따옴표로 감싸야 합니다"]                                 

user_answer = st.radio(question, options, index=0)
if user_answer:
    if user_answer == "아래 선택을 고르세요...": # 이 옵션은 streamlit이 1.26.0 이상으로 업그레이드될 때까지의 임시 방편입니다. 그래서 index=None을 사용할 수 있습니다.
        ''
    else:
        answer = '4195b7d7c6d53157c16bb34de5c5352e'
        # 옵션 문자 (A, B, C, 또는 D)를 추출하세요
        selected_option = user_answer.split(')')[0] + ')'
        response = session.sql(f"call common_db.resources.quiz_temp('{answer}', '{user_answer}', 'False')").collect()
        if response:
            value = response[0]['QUIZ_TEMP']
        st.write(f"{selected_option} {value}")

## 다음 단계

실습 단계를 완료하고 **지식 테스트** 질문에 정답을 입력하셨다면, Snowflake 강사의 안내에 따라 다음 Notebook으로 진행하세요.
